# 02 🧠 文脈と抑制: pair / refractory / history

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kagioneko/cse-frog/blob/main/notebooks/02_context_and_inhibition.ipynb)

> 解説は [HANDBOOK](../docs/HANDBOOK.md) と [CONFIG_GUIDE](../docs/CONFIG_GUIDE.md)、全設定は [REFERENCE](../docs/REFERENCE.md) にあります。

In [1]:
# 🐸 の準備(Colab では最初に1回だけインストールします)
try:
    import cse
except ImportError:
    %pip install -q git+https://github.com/kagioneko/cse-frog.git
from cse import Frog, END

## 1. Pair Context ON/OFF
「右」の次は、その前が「右」なら「下」、「左」なら「上」。直前1つだけでは決まりません。

In [2]:
data = [["右", "右", "下"]] * 3 + [["左", "右", "上"]] * 3
for cap in [2048, 0]:
    f = Frog(pair_context_capacity=cap).learn(data)
    print(f"capacity={cap:<5} 右右→{f.top(['右','右'])}   左右→{f.top(['左','右'])}")

capacity=2048  右右→[('下', 0.6350177732408689), ('上', 0.18314682927755285), ('右', 0.1818353974815782)]   左右→[('上', 0.6373229005670438), ('下', 0.18138851931581088), ('右', 0.18128858011714544)]
capacity=0     右右→[('上', 0.33487081157306653), ('下', 0.3326562356187337), ('右', 0.3324729528081997)]   左右→[('上', 0.33487081157306653), ('下', 0.3326562356187337), ('右', 0.3324729528081997)]


capacity=0 だと直前2つの並びを覚えられず、どちらも半々になります。`show()` の「並び」列を見比べてください。

In [3]:
Frog().learn(data).show(["左", "右"])
Frog(pair_context_capacity=0).learn(data).show(["左", "右"])

🐸 ['左', '右'] の次の候補
候補            直接      履歴      並び      痕跡      合計点      確率
上          0.887   0.000   1.000   0.000    1.887   0.637
下          0.882   0.000   0.000   0.000    0.882   0.181
右          0.881   0.000   0.000   0.000    0.881   0.181
🐸 ['左', '右'] の次の候補
候補            直接      履歴      並び      痕跡      合計点      確率
上          0.887   0.000   0.000   0.000    0.887   0.335
下          0.882   0.000   0.000   0.000    0.882   0.333
右          0.881   0.000   0.000   0.000    0.881   0.332


## 2. Refractory Ladder(不応期)
最近出た記号を何個前まで禁止するか。必要な繰り返しまで消してしまう様子を見ます。

In [4]:
data = [["右", "右", "下", "右"]] * 3
for steps in [0, 1, 2, 5]:
    f = Frog(refractory_steps=steps).learn(data)
    print(f"refractory_steps={steps}  右右→{f.top(['右','右'], k=2)}   右右下→{f.top(['右','右','下'], k=2)}")

refractory_steps=0  右右→[('下', 0.6356594353754365), (<END>, 0.18232239201565928)]   右右下→[('右', 1.0)]
refractory_steps=1  右右→[('下', 0.7771070384324238), (<END>, 0.22289296156757624)]   右右下→[('右', 1.0)]
refractory_steps=2  右右→[('下', 0.7771070384324238), (<END>, 0.22289296156757624)]   右右下→[(<END>, 1.0)]
refractory_steps=5  右右→[('下', 0.7771070384324238), (<END>, 0.22289296156757624)]   右右下→[(<END>, 1.0)]


In [5]:
Frog(refractory_steps=2).learn(data).show(["右", "右", "下"])

🐸 ['右', '右', '下'] の次の候補
候補            直接      履歴      並び      痕跡      合計点      確率
(点数がプラスの候補が1つもありません → エンジンは <END> を確率1で返します)
右          0.891   0.000   1.000   0.000    0.000   0.000  ← 不応期で消された


## 3. max_consecutive_repeats(不応期の例外)
自分自身へのつながりを覚えていれば、何回まで連続を許すか。**不応期がONのときだけ** 効きます。

In [6]:
data = [["A", "A", "A", "B"]] * 4
for m in [1, 2, 3]:
    f = Frog(refractory_steps=2, max_consecutive_repeats=m).learn(data)
    print(f"max_consecutive_repeats={m}  AA→{f.top(['A','A'], k=2)}")
# 「AA」の時点で A は2回続いている → 3回目を許すのは max_consecutive_repeats=3 から
f = Frog(refractory_steps=0, max_consecutive_repeats=1).learn(data)
print("refractory_steps=0 なら:", f.top(["A", "A"], k=2))

max_consecutive_repeats=1  AA→[('B', 1.0)]
max_consecutive_repeats=2  AA→[('B', 1.0)]
max_consecutive_repeats=3  AA→[('A', 0.8140595898515445), ('B', 0.18594041014845555)]
refractory_steps=0 なら: [('A', 0.8140595898515445), ('B', 0.18594041014845555)]


## 4. history_boost(活性の残りを点数に足す)
正なら最近の記号を押し上げ、**負にすると候補が全滅** することがあります。

In [7]:
data = [["右", "右", "下", "左"]] * 4
for hb in [0.0, 0.35, 1.0, -0.3]:
    f = Frog(history_boost=hb).learn(data)
    print(f"history_boost={hb:<5} 右右→{f.top(['右','右'], k=3)}")

history_boost=0.0   右右→[('下', 0.7774277762842404), ('右', 0.22257222371575963)]
history_boost=0.35  右右→[('右', 0.5355705721822609), ('下', 0.4633925907511441), ('左', 0.0010368370665951251)]
history_boost=1.0   右右→[('右', 0.9391401675082017), ('下', 0.06085957393968631), ('左', 2.585521120082222e-07)]
history_boost=-0.3  右右→[(<END>, 1.0)]


In [8]:
Frog(history_boost=-0.3).learn(data).show(["右", "右"])

🐸 ['右', '右'] の次の候補
候補            直接      履歴      並び      痕跡      合計点      確率
(点数がプラスの候補が1つもありません → エンジンは <END> を確率1で返します)
